# Import Library

In [19]:
import pandas as pd
from pathlib import Path

# Load Data

Baca file raw

In [20]:
file_path = "./data/raw/eur_idr_history_data.csv"
# text = Path(file_path).read_text(encoding="utf-8-sig")

Bersihkan quote luar tiap baris

In [ ]:
# clean_lines = []

# for line in text.splitlines():
#     line = line.strip()
    
#     if line.startswith('"') and line.endswith('"'):
#         line = line[1:-1]
    
#     line = line.replace('""', '"')
#     clean_lines.append(line)

# clean_text = "\n".join(clean_lines)


Load clean data

In [21]:
# 3. Load ulang sebagai CSV normal
df = pd.read_csv(file_path)

df.head()

,Date,Price,Open,High,Low,Vol.,Change %
0,01/30/2026,"19,890.2","20,047.6","20,066.5","19,889.4",NaN,-0.79%
1,01/29/2026,"20,047.6","19,969.2","20,149.2","19,939.6",NaN,0.39%
2,01/28/2026,"19,969.2","20,186.7","20,194.3","19,871.4",NaN,-1.08%
3,01/27/2026,"20,187.6","19,922.0","20,250.4","19,866.5",NaN,1.30%
4,01/26/2026,"19,929.5","19,897.2","20,007.3","19,853.2",NaN,0.20%


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1326 entries, 0 to 1325
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Date      1326 non-null   object
 1   Price     1326 non-null   object
 2   Open      1326 non-null   object
 3   High      1326 non-null   object
 4   Low       1326 non-null   object
 5   Vol.      538 non-null    object
 6   Change %  1326 non-null   object
dtypes: object(7)
memory usage: 72.6+ KB


# Data preprocessing

## Rename column

In [23]:
df = df.rename(columns=
{
    "Date": "timestamp",
    "Price": "close",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Vol.": "volume"
})

In [24]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1326 entries, 0 to 1325
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   timestamp  1326 non-null   object
 1   close      1326 non-null   object
 2   open       1326 non-null   object
 3   high       1326 non-null   object
 4   low        1326 non-null   object
 5   volume     538 non-null    object
 6   Change %   1326 non-null   object
dtypes: object(7)
memory usage: 72.6+ KB
None


## Bersihkan angka ribuan

In [25]:
price_cols = ["open", "high", "low", "close"]
for col in price_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

## Convert timestamp

In [26]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1326 entries, 0 to 1325
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  1326 non-null   datetime64[ns]
 1   close      1326 non-null   float64       
 2   open       1326 non-null   float64       
 3   high       1326 non-null   float64       
 4   low        1326 non-null   float64       
 5   volume     538 non-null    object        
 6   Change %   1326 non-null   object        
dtypes: datetime64[ns](1), float64(4), object(2)
memory usage: 72.6+ KB


## Isi dengan dummy data

In [28]:
# Volume tidak tersedia, isi dummy
df["volume"] = 0.0
df["amount"] = 0.0

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1326 entries, 0 to 1325
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  1326 non-null   datetime64[ns]
 1   close      1326 non-null   float64       
 2   open       1326 non-null   float64       
 3   high       1326 non-null   float64       
 4   low        1326 non-null   float64       
 5   volume     1326 non-null   float64       
 6   Change %   1326 non-null   object        
 7   amount     1326 non-null   float64       
dtypes: datetime64[ns](1), float64(6), object(1)
memory usage: 83.0+ KB


## Sort lama ke baru

In [30]:
df = df.sort_values("timestamp").reset_index(drop=True)

In [31]:
df.head()

,timestamp,close,open,high,low,volume,Change %,amount
0,2021-01-01,17246.3,17246.3,17246.3,17246.3,0.0,0.58%,0.0
1,2021-01-04,17006.3,17184.6,17219.7,16980.1,0.0,-1.39%,0.0
2,2021-01-05,17091.4,17017.2,17125.2,17015.1,0.0,0.50%,0.0
3,2021-01-06,17107.1,17109.9,17184.2,17042.9,0.0,0.09%,0.0
4,2021-01-07,17043.0,17127.7,17190.0,17032.1,0.0,-0.37%,0.0


## Ambil kolom untuk Kronos

In [32]:
kronos_df = df[["timestamp", "open", "high", "low", "close", "volume", "amount"]].copy()

In [33]:
kronos_df.head()

,timestamp,open,high,low,close,volume,amount
0,2021-01-01,17246.3,17246.3,17246.3,17246.3,0.0,0.0
1,2021-01-04,17184.6,17219.7,16980.1,17006.3,0.0,0.0
2,2021-01-05,17017.2,17125.2,17015.1,17091.4,0.0,0.0
3,2021-01-06,17109.9,17184.2,17042.9,17107.1,0.0,0.0
4,2021-01-07,17127.7,17190.0,17032.1,17043.0,0.0,0.0


## Hapus missing value kalau ada

In [34]:
kronos_df = kronos_df.dropna()
kronos_df.head()

,timestamp,open,high,low,close,volume,amount
0,2021-01-01,17246.3,17246.3,17246.3,17246.3,0.0,0.0
1,2021-01-04,17184.6,17219.7,16980.1,17006.3,0.0,0.0
2,2021-01-05,17017.2,17125.2,17015.1,17091.4,0.0,0.0
3,2021-01-06,17109.9,17184.2,17042.9,17107.1,0.0,0.0
4,2021-01-07,17127.7,17190.0,17032.1,17043.0,0.0,0.0


In [35]:
kronos_df.to_csv("./data/preprocessing/eur_idr_kronos_clean.csv", index=False)